In [1]:
import pandas as pd
import numpy as np

In [2]:
# Load engineered feature matrix and isolate future fixtures
print ("Loading engineered feature matrix...")
df = pd.read_csv('../data/processed/training_features.csv')
df['date'] = pd.to_datetime(df['date'])

future_fixtures= df[df['outcome'].isna().copy()]
future_fixtures.to_csv('../data/processed/future_fixtures.csv', index=False)
print(f"Isolated {len(future_fixtures)} future fixtures for simulation.")

historical_df = df[df['outcome'].notna().copy()]

Loading engineered feature matrix...
Isolated 72 future fixtures for simulation.


In [3]:
# Define features and target variable
features=[
    'home_team', 'away_team',
    'rank_diff', 'point_diff',
    'home_fifa_rank',
    'home_recent_form', 'away_recent_form',
    'h2h_goal_diff',
    'match_weight', 'is_neutral'
]

target = 'outcome'

historical_df['home_team'] = historical_df['home_team'].astype('category')
historical_df['away_team'] = historical_df['away_team'].astype('category')

historical_df['outcome'] = historical_df['outcome'].astype(int)

C:\Users\mohamed\AppData\Local\Temp\ipykernel_23308\4253756444.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  historical_df['home_team'] = historical_df['home_team'].astype('category')
C:\Users\mohamed\AppData\Local\Temp\ipykernel_23308\4253756444.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  historical_df['away_team'] = historical_df['away_team'].astype('category')
C:\Users\mohamed\AppData\Local\Temp\ipykernel_23308\4253756444.py:16: SettingWithCopyWarning: 
A value is trying to be set on a co

In [4]:
# Split the historical data into training and validation sets based on date
train_mask = historical_df['date'] <= '2021-12-31'
val_mask = (historical_df['date'] >= '2022-1-1') & (historical_df['date'] < '2025-12-31')

train_data = historical_df[train_mask]
val_data = historical_df[val_mask]

X_train = train_data[features]
y_train = train_data[target]

X_val = val_data[features]
y_val = val_data[target]

print(f"Training Matrix Shape:   {X_train.shape} (Matches from 1993 - 2021)")
print(f"Validation Matrix Shape: {X_val.shape} (Matches from 2022 - 2025)")

Training Matrix Shape:   (23077, 10) (Matches from 1993 - 2021)
Validation Matrix Shape: (3786, 10) (Matches from 2022 - 2025)


In [5]:
# Initialize and train the XGBoost multi-class classifier
from xgboost import XGBClassifier
from sklearn.metrics import log_loss
import joblib
print("Initializing XGBoost Multi-Class Classifier...")

model = XGBClassifier(
    n_estimators=1000,
    learning_rate=0.01,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    objective='multi:softprob',
    num_class=3,
    eval_metric='mlogloss',
    enable_categorical=True,
    early_stopping_rounds=50,
    random_state=42
)

model.fit(
    X_train, y_train,
    eval_set=[(X_train, y_train), (X_val, y_val)],
    verbose=100
)


val_probs = model.predict_proba(X_val)
final_val_loss = log_loss(y_val, val_probs)

print("\n==========================================")
print(f"Model Training Complete!")
print(f"Final Validation Log Loss: {final_val_loss:.4f}")
print("==========================================")

joblib.dump(model, '../outputs/models/xgboost_model.pkl')

Initializing XGBoost Multi-Class Classifier...
[0]	validation_0-mlogloss:1.04697	validation_1-mlogloss:1.05246
[100]	validation_0-mlogloss:0.89074	validation_1-mlogloss:0.94260
[200]	validation_0-mlogloss:0.82460	validation_1-mlogloss:0.91747
[300]	validation_0-mlogloss:0.78521	validation_1-mlogloss:0.91100
[400]	validation_0-mlogloss:0.75764	validation_1-mlogloss:0.91119
[402]	validation_0-mlogloss:0.75722	validation_1-mlogloss:0.91121

Model Training Complete!
Final Validation Log Loss: 0.9106


['../outputs/models/xgboost_model.pkl']